# Lab 5 of 8: Unsupervised learning II, anomaly detection and limited labels
Learn when labels are absent, rare or expensive.

Run the cells from top to bottom. Keep this notebook inside `all_experiments/`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.semi_supervised import SelfTrainingClassifier

PROJECT_ROOT = Path.cwd().parent
DATA = PROJECT_ROOT / "all_datasets"

## Part A: Isolation Forest fraud ranking

In [ ]:
fraud = pd.read_csv(DATA / "credit_card_fraud_dataset.csv")
print(fraud.head())

feature_cols = [
    "amount", "transaction_hour", "foreign_transaction", "location_mismatch",
    "device_trust_score", "velocity_last_24h", "cardholder_age",
]
X = fraud[feature_cols]
y = fraud["is_fraud"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

forest = IsolationForest(n_estimators=300, contamination="auto", random_state=42, n_jobs=-1)
forest.fit(X_train)
anomaly_score = -forest.decision_function(X_test)

In [ ]:
prevalence = y_test.mean()
ap = average_precision_score(y_test, anomaly_score)
print("fraud prevalence:", round(prevalence, 4))
print("ROC AUC:", round(roc_auc_score(y_test, anomaly_score), 4))
print("average precision:", round(ap, 4), " (", round(ap / prevalence, 1), "x prevalence)")

ranked = X_test.copy()
ranked["fraud_label"] = y_test.to_numpy()
ranked["anomaly_score"] = anomaly_score
ranked = ranked.sort_values("anomaly_score", ascending=False)

review_budget = 100
top_k_precision = ranked.head(review_budget)["fraud_label"].mean()
print(f"fraud found in the top {review_budget}: {top_k_precision:.0%}")
print(ranked[["fraud_label", "anomaly_score", "amount"]].head(10))

In [ ]:
hits = ranked["fraud_label"].to_numpy()
k_values = np.arange(10, 1001, 10)
precision_at_k = [hits[:k].mean() for k in k_values]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].hist(anomaly_score[y_test.to_numpy() == 0], bins=30, alpha=0.65, label="non-fraud")
axes[0].hist(anomaly_score[y_test.to_numpy() == 1], bins=30, alpha=0.65, label="fraud")
axes[0].set_xlabel("Anomaly score")
axes[0].set_ylabel("Transactions")
axes[0].set_title("Isolation Forest score distribution")
axes[0].legend()
axes[1].plot(k_values, precision_at_k)
axes[1].axhline(prevalence, linestyle="--", color="grey", label=f"prevalence {prevalence:.1%}")
axes[1].set_xlabel("Number of top-ranked cases reviewed (k)")
axes[1].set_ylabel("Fraud rate among those k")
axes[1].set_title("Precision falls as the review budget grows")
axes[1].legend()
plt.tight_layout()
plt.show()

## Part B: Self-training and active learning with few labels

In [ ]:
rng = np.random.default_rng(42)
cancer = pd.read_csv(DATA / "breast_cancer_dataset.csv")
X_c = cancer.drop(columns=["id", "diagnosis", "Unnamed: 32"], errors="ignore")
y_c = (cancer["diagnosis"] == "M").astype(int).to_numpy()

X_dev, Xc_test, y_dev, yc_test = train_test_split(
    X_c, y_c, test_size=0.25, random_state=42, stratify=y_c
)
X_pool, X_val, y_pool, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=7, stratify=y_dev
)
X_pool = X_pool.reset_index(drop=True)
y_pool = np.asarray(y_pool)
print("pool:", len(X_pool), " validation:", len(X_val), " test:", len(Xc_test))

def make_model():
    return Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000)),
    ])

In [ ]:
y_partial = y_pool.copy()
y_partial[rng.random(len(y_partial)) < 0.70] = -1
print("labels kept:", int((y_partial != -1).sum()), "of", len(y_partial))

semi = Pipeline([
    ("scale", StandardScaler()),
    ("model", SelfTrainingClassifier(LogisticRegression(max_iter=2000), threshold=0.90)),
])
semi.fit(X_pool, y_partial)
print("self-training validation accuracy:",
      round(accuracy_score(y_val, semi.predict(X_val)), 3))

In [ ]:
class0 = np.where(y_pool == 0)[0]
class1 = np.where(y_pool == 1)[0]
initial = np.concatenate([rng.choice(class0, 20, replace=False),
                          rng.choice(class1, 20, replace=False)])
active_labeled = set(initial.tolist())
random_labeled = set(initial.tolist())
rng_random = np.random.default_rng(123)

query_batch, rounds = 15, 8
label_counts, active_scores, random_scores = [], [], []

for round_no in range(rounds):
    active_idx, random_idx = sorted(active_labeled), sorted(random_labeled)
    active = make_model().fit(X_pool.iloc[active_idx], y_pool[active_idx])
    random_model = make_model().fit(X_pool.iloc[random_idx], y_pool[random_idx])

    label_counts.append(len(active_labeled))
    active_scores.append(accuracy_score(y_val, active.predict(X_val)))
    random_scores.append(accuracy_score(y_val, random_model.predict(X_val)))
    print(f"round {round_no + 1}: labels={label_counts[-1]:3d} "
          f"uncertainty={active_scores[-1]:.3f} random={random_scores[-1]:.3f}")

    if round_no == rounds - 1:
        break

    unlabeled = np.array([i for i in range(len(X_pool)) if i not in active_labeled])
    uncertainty = np.abs(active.predict_proba(X_pool.iloc[unlabeled])[:, 1] - 0.5)
    active_labeled.update(unlabeled[np.argsort(uncertainty)[:query_batch]].tolist())

    unlabeled = np.array([i for i in range(len(X_pool)) if i not in random_labeled])
    random_labeled.update(rng_random.choice(unlabeled, query_batch, replace=False).tolist())

plt.figure(figsize=(6, 3.6))
plt.plot(label_counts, active_scores, marker="o", label="uncertainty sampling")
plt.plot(label_counts, random_scores, marker="o", label="random querying")
plt.xlabel("Number of labelled pool examples")
plt.ylabel("Validation accuracy")
plt.title("Label efficiency during development")
plt.legend()
plt.tight_layout()
plt.show()

Both policies are finished, so the test set is opened **once** for each. Do not rerun the next cell while tweaking things.

In [ ]:
active_idx, random_idx = sorted(active_labeled), sorted(random_labeled)
active_final = make_model().fit(X_pool.iloc[active_idx], y_pool[active_idx])
random_final = make_model().fit(X_pool.iloc[random_idx], y_pool[random_idx])
print("final test accuracy, uncertainty sampling:",
      round(accuracy_score(yc_test, active_final.predict(Xc_test)), 3))
print("final test accuracy, random querying:     ",
      round(accuracy_score(yc_test, random_final.predict(Xc_test)), 3))

## Try it yourself

In [ ]:
# Part 1: fraud precision at different review budgets.
print("fraud cases in the test set:", int(y_test.sum()), "of", len(y_test))
for budget in [50, 100, 300]:
    found = int(ranked.head(budget)["fraud_label"].sum())
    print(f"review budget {budget:3d}: precision = {found / budget:.0%} ({found} frauds found)")

In [ ]:
# Part 2: start with ten labels per class and reassess uncertainty sampling.
def run_active_vs_random(n_per_class, rounds=8, query_batch=15):
    rng_start = np.random.default_rng(42)
    rng_rand = np.random.default_rng(123)
    start = np.concatenate([rng_start.choice(class0, n_per_class, replace=False),
                            rng_start.choice(class1, n_per_class, replace=False)])
    a_set, r_set = set(start.tolist()), set(start.tolist())
    counts, a_scores, r_scores = [], [], []
    for round_no in range(rounds):
        a_idx, r_idx = sorted(a_set), sorted(r_set)
        a_model = make_model().fit(X_pool.iloc[a_idx], y_pool[a_idx])
        r_model = make_model().fit(X_pool.iloc[r_idx], y_pool[r_idx])
        counts.append(len(a_set))
        a_scores.append(accuracy_score(y_val, a_model.predict(X_val)))
        r_scores.append(accuracy_score(y_val, r_model.predict(X_val)))
        if round_no == rounds - 1:
            break
        unl = np.array([i for i in range(len(X_pool)) if i not in a_set])
        unc = np.abs(a_model.predict_proba(X_pool.iloc[unl])[:, 1] - 0.5)
        a_set.update(unl[np.argsort(unc)[:query_batch]].tolist())
        unl = np.array([i for i in range(len(X_pool)) if i not in r_set])
        r_set.update(rng_rand.choice(unl, query_batch, replace=False).tolist())
    return counts, a_scores, r_scores

counts10, active10, random10 = run_active_vs_random(10)
for n, a, r in zip(counts10, active10, random10):
    print(f"labels={n:3d} uncertainty={a:.3f} random={r:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
axes[0].plot(label_counts, active_scores, marker="o", label="uncertainty sampling")
axes[0].plot(label_counts, random_scores, marker="o", label="random querying")
axes[0].set_title("Start with 20 labels per class")
axes[1].plot(counts10, active10, marker="o", label="uncertainty sampling")
axes[1].plot(counts10, random10, marker="o", label="random querying")
axes[1].set_title("Start with 10 labels per class")
for ax in axes:
    ax.set_xlabel("Number of labelled pool examples")
    ax.legend()
axes[0].set_ylabel("Validation accuracy")
plt.tight_layout()
plt.show()